# Genomic VEP — Training Notebook

**Setup:**
1. Runtime → Change runtime type → **T4 GPU**
2. Have parquet files in Google Drive (genomic_vep folder)
3. Run all cells

## 1. Clone repo & install deps

In [ ]:
import os

# Clone repo (skip if already cloned)
if not os.path.exists("genomic-vep"):
    !git clone https://github.com/theomalaper/genomic-vep.git

%cd genomic-vep
!pip install -q -r requirements.txt

## 2. Verify GPU & copy data

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Runtime → Change runtime type → T4 GPU")
    
# Copy parquet files from Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p data
!cp /content/drive/MyDrive/genomic_vep/*.parquet data/

for split in ["train", "val", "test"]:
    path = f"data/{split}.parquet"
    if os.path.exists(path):
        print(f"  {split}.parquet: {os.path.getsize(path) / 1e6:.1f} MB")
    else:
        print(f"  MISSING: {path}")

## 4. Train

In [ ]:
from src.model.train import train

model = train(epochs=10, batch_size=32, lr=1e-3, pos_weight=2.0)

## 5. Test evaluation

In [ ]:
from src.model.train import train

model = train(epochs=5, batch_size=32, lr=1e-3, pos_weight=2.0)

## 6. Download checkpoint

In [ ]:
from google.colab import files
files.download("checkpoints/best.pth")

## 5. Download checkpoint

In [ ]:
from google.colab import files
files.download("checkpoints/best_model.pth")